# 09. COUNT · DISTINCT · LIMIT

*Module: 3 — DML · Companion summary: `lectures/09-count-distinct-limit.md` · SQL engine: SQLite (in-memory)*

## What you will study

- count rows matching query criteria
- remove duplicate values from a result set
- peek at a result set with LIMIT instead of retrieving everything

---
## Setup — build the sample database this notebook runs on
*Run the cell below first: it creates an in-memory SQLite database and the `q()` helper that prints every query's result set.*

In [ ]:
import sqlite3
import textwrap

conn = sqlite3.connect(':memory:')  # throw-away in-memory database
conn.execute("PRAGMA foreign_keys = ON")

def q(sql):
    """Run a query, print formatted SQL, and display the result set as a table."""
    clean_sql = textwrap.dedent(sql).strip()

    # 1. Print formatted SQL query
    print("\n" + "=" * 60)
    print("QUERY:")
    print(textwrap.indent(clean_sql, "  "))
    print("-" * 60)

    cur = conn.execute(sql)
    rows = cur.fetchall()

    if cur.description:  # SELECT / statement returning rows
        headers = [d[0] for d in cur.description]
        str_rows = [[("" if v is None else str(v)) for v in r] for r in rows]

        # Calculate dynamic column widths for clean alignment
        widths = [
            max(len(h), *(len(r[i]) for r in str_rows), 1)
            for i, h in enumerate(headers)
        ] if str_rows else [len(h) for h in headers]

        header_line = " | ".join(h.ljust(widths[i]) for i, h in enumerate(headers))
        sep_line = "-+-".join("-" * widths[i] for i in range(len(headers)))

        print(header_line)
        print(sep_line)
        for r in str_rows:
            print(" | ".join(val.ljust(widths[i]) for i, val in enumerate(r)))
        if not str_rows:
            print("(0 rows returned)")
    else:  # DDL / DML (CREATE, INSERT, UPDATE, DELETE)
        affected = cur.rowcount
        print(f"Done (rows affected: {affected})" if affected != -1 else "Done.")

    print("=" * 60)

print('Connection ready:', conn)

## 🧪 Sample data — the MEDALS table (countries can win the same medal repeatedly)

In [ ]:
q("CREATE TABLE medals (country TEXT, medaltype TEXT, year INTEGER, athlete TEXT)")
q("""
INSERT INTO medals VALUES
    ('CANADA','GOLD',2018,'Smith'),  ('CANADA','GOLD',2016,'Tremblay'),
    ('CANADA','SILVER',2018,'Lee'),  ('CHINA','GOLD',2018,'Wang'),
    ('CHINA','GOLD',2016,'Li'),      ('UK','GOLD',2018,'Brown'),
    ('UK','SILVER',2016,'Cox'),      ('USA','GOLD',2018,'Davis'),
    ('USA','SILVER',2018,'Miller'),  ('USA','BRONZE',2016,'Taylor'),
    ('UK','BRONZE',2018,'Wilson'),   ('CHINA','BRONZE',2018,'Zhang'),
    ('CANADA','BRONZE',2016,'Clark')
""")
q("SELECT * FROM medals")

## 🧪 COUNT — how many rows?

In [ ]:
q("SELECT COUNT(*) FROM medals")
q("SELECT COUNT(*) FROM medals WHERE medaltype = 'GOLD'")
q("SELECT COUNT(country) FROM medals WHERE country = 'CANADA'")

## 🧪 DISTINCT — unique values in a column

In [ ]:
q("SELECT country FROM medals")   # with duplicates
q("SELECT DISTINCT country FROM medals")   # each country once
q("SELECT DISTINCT country FROM medals WHERE medaltype = 'GOLD'")

## 🧪 LIMIT — restrict how many rows come back

In [ ]:
q("SELECT * FROM medals LIMIT 5")
q("SELECT * FROM medals WHERE year = 2018 LIMIT 3")

## 🧪 Combine all three: distinct gold countries + their counts

In [ ]:
q("SELECT DISTINCT country FROM medals WHERE medaltype = 'GOLD' ORDER BY country LIMIT 4")

## 💡 Tips & tricks

- **Explore with LIMIT, not SELECT *** — on millions of rows, LIMIT answers almost
  instantly; SELECT * walks the whole table through the wire.
- 📕 Dialect note: `LIMIT 10` (SQLite/MySQL/Postgres) = `TOP 10` (SQL Server) =
  `FETCH FIRST 10 ROWS ONLY` (DB2/Oracle 12c+).

## 🎯 Your turn

1. How many medals did the UK win in total? In gold?
2. The list of medal *types* awarded in 2018 (each once).

In [ ]:
# 💡 Solution
q("SELECT COUNT(*) AS uk_total FROM medals WHERE country = 'UK'")
q("SELECT COUNT(*) AS uk_gold FROM medals WHERE country = 'UK' AND medaltype = 'GOLD'")
q("SELECT DISTINCT medaltype FROM medals WHERE year = 2018")